In [1]:
!pip install PyPDF2 pdfplumber spacy streamlit
!python -m spacy download en_core_web_sm

import pandas as pd
import PyPDF2
import pdfplumber
import re
import spacy
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
print("Setup complete! NLP ready!")
nlp = spacy.load("en_core_web_sm")
print("spaCy loaded!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Re

In [2]:
test_resumes = {
    'ml_engineer.txt': """
Anchit Shrivastava
ML Engineer | Bareli, MP | 3 years exp
Email: anchit@email.com | LinkedIn: linkedin.com/in/anchit

SKILLS:
Python, SQL, Power BI, scikit-learn, XGBoost, Pandas, Streamlit
Machine Learning, NLP, Data Visualization

PROJECTS:
1. Heart Disease Prediction (81% accuracy) - Random Forest
2. Customer Churn Prediction (79%) - XGBoost
3. Sales Dashboard - Power BI

EXPERIENCE:
ML Intern | XYZ Tech | 6 months
• Built 81% heart disease classifier
• Deployed Streamlit web app

EDUCATION:
B.Tech Computer Science | 2025
""",

    'data_analyst.txt': """
Priya Sharma
Data Analyst | Indore | 2 years exp

SKILLS:
SQL, Power BI, Excel, Tableau, Python (pandas)
Data Cleaning, Visualization, Dashboarding

PROJECTS:
1. Sales Performance Dashboard (Power BI)
2. Customer Segmentation Analysis
3. SQL Query Optimization (50x faster)

EXPERIENCE:
Data Analyst | ABC Corp | 18 months
• Built 10+ Power BI dashboards
• SQL queries for 1M+ rows

EDUCATION:
B.Sc Statistics | 2024
""",

    'software_dev.txt': """
Rahul Kumar
Software Developer | Bhopal | 4 years exp

SKILLS:
Java, Spring Boot, React, AWS, Docker, Kubernetes
Microservices, REST API, Database Design

PROJECTS:
1. E-commerce Platform (Java + React)
2. Microservices Architecture
3. AWS Cloud Deployment

EXPERIENCE:
SDE II | TechCorp | 2 years
• Led 5-member team
• Deployed 10+ microservices

EDUCATION:
B.Tech IT | 2022
"""
}

print("3 TEST RESUMES LOADED!")
for filename, text in test_resumes.items():
    print(f"\n {filename}:")
    print(text[:150] + "...")







3 TEST RESUMES LOADED!

 ml_engineer.txt:

Anchit Shrivastava
ML Engineer | Bareli, MP | 3 years exp
Email: anchit@email.com | LinkedIn: linkedin.com/in/anchit

SKILLS:
Python, SQL, Power BI, ...

 data_analyst.txt:

Priya Sharma
Data Analyst | Indore | 2 years exp

SKILLS:
SQL, Power BI, Excel, Tableau, Python (pandas)
Data Cleaning, Visualization, Dashboarding

...

 software_dev.txt:

Rahul Kumar
Software Developer | Bhopal | 4 years exp

SKILLS:
Java, Spring Boot, React, AWS, Docker, Kubernetes
Microservices, REST API, Database De...


In [3]:
def extract_text(filename,text_content):
  text = re.sub(r'\s+',' ',text_content)
  text = text.strip()
  text = text.lower()
  return text

cleaned_resumes = {}
for filename, content in test_resumes.items():
  cleaned = extract_text(filename,content)
  cleaned_resumes[filename] = cleaned
  print(f"\n {filename} (Cleaned):")
  print(cleaned[:300] + "..." if len(cleaned) >300 else cleaned )


 ml_engineer.txt (Cleaned):
anchit shrivastava ml engineer | bareli, mp | 3 years exp email: anchit@email.com | linkedin: linkedin.com/in/anchit skills: python, sql, power bi, scikit-learn, xgboost, pandas, streamlit machine learning, nlp, data visualization projects: 1. heart disease prediction (81% accuracy) - random forest ...

 data_analyst.txt (Cleaned):
priya sharma data analyst | indore | 2 years exp skills: sql, power bi, excel, tableau, python (pandas) data cleaning, visualization, dashboarding projects: 1. sales performance dashboard (power bi) 2. customer segmentation analysis 3. sql query optimization (50x faster) experience: data analyst | a...

 software_dev.txt (Cleaned):
rahul kumar software developer | bhopal | 4 years exp skills: java, spring boot, react, aws, docker, kubernetes microservices, rest api, database design projects: 1. e-commerce platform (java + react) 2. microservices architecture 3. aws cloud deployment experience: sde ii | techcorp | 2 years • led...

In [4]:

skills_list = [

    'python', 'sql', 'r', 'java', 'c++', 'javascript', 'scala', 'go', 'ruby',


    'machine learning', 'deep learning', 'nlp', 'tensorflow', 'pytorch',
    'scikit-learn', 'xgboost', 'keras', 'huggingface','Ethical Hacking',


    'pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly', 'tableau', 'power bi',


    'spark', 'hadoop', 'kafka', 'airflow', 'dbt',


    'aws', 'azure', 'gcp', 'docker', 'kubernetes',


    'postgresql', 'mysql', 'mongodb', 'redshift', 'snowflake',


    'looker', 'metabase','Cyber Security','Networking',


    'jenkins', 'gitlab', 'circleci', 'terraform','Linux'
]

print(f"{len(skills_list)} skills loaded!")
print("Sample:", skills_list[:10])


50 skills loaded!
Sample: ['python', 'sql', 'r', 'java', 'c++', 'javascript', 'scala', 'go', 'ruby', 'machine learning']


In [5]:
def extract_skills(resume_text, skills_list):
    found_skills = []
    text = resume_text.lower()

    for skill in skills_list:
        pattern = r'\b' + re.escape(skill.lower()) + r'\b'
        if re.search(pattern, text):
            found_skills.append(skill)

    return found_skills


print("\n" + "="*50)
print("Skills Extraction Test")
print("="*50)

for filename, text in cleaned_resumes.items():
    skills = extract_skills(text, skills_list)
    print(f"Skills Found: {skills}")
    print(f"Count: {len(skills)}")



Skills Extraction Test
Skills Found: ['python', 'sql', 'machine learning', 'nlp', 'scikit-learn', 'xgboost', 'pandas', 'power bi']
Count: 8
Skills Found: ['python', 'sql', 'pandas', 'tableau', 'power bi']
Count: 5
Skills Found: ['java', 'aws', 'docker', 'kubernetes']
Count: 4


In [6]:
def extract_experience(resume_text):

    text = resume_text.lower()
    patterns = [
        r'(\d+)\s*years?',
        r'(\d+)\s*yrs?',
        r'experience[:\s]*(\d+)'
    ]

    max_exp = 0
    for pattern in patterns:
        matches = re.findall(pattern, text)
        for match in matches:
            try:
                years = int(match)
                if years > max_exp:
                    max_exp = years
            except:
                pass
    return max_exp
print("\n" + "="*60)
print("TEST RESULTS - SKILLS + EXPERIENCE")
print("="*60)

for filename, text in cleaned_resumes.items():
    skills = extract_skills(text,skills_list)
    exp = extract_experience(text)

    print(f"\n {filename}:")
    print(f"   Skills: {', '.join(skills)}")
    print(f"   Experience: {exp} years")
    print(f"   Total skills: {len(skills)}")



TEST RESULTS - SKILLS + EXPERIENCE

 ml_engineer.txt:
   Skills: python, sql, machine learning, nlp, scikit-learn, xgboost, pandas, power bi
   Experience: 3 years
   Total skills: 8

 data_analyst.txt:
   Skills: python, sql, pandas, tableau, power bi
   Experience: 2 years
   Total skills: 5

 software_dev.txt:
   Skills: java, aws, docker, kubernetes
   Experience: 4 years
   Total skills: 4


In [7]:
def extract_name(resume_text):

    lines = resume_text.split('\n')
    first_line_original = lines[0].strip()  # Original case


    first_line_lower = first_line_original.lower()

    name_patterns = [
        r'^([a-z]+\s+[a-z]+)',
        r'^([a-z\s]+?)\s+(ml|data|software)',  ]

    for pattern in name_patterns:
        match = re.search(pattern, first_line_lower)
        if match:
            name_lower = match.group(1).strip()
            # Title case banao
            name = name_lower.title()
            return name

    return "Name not found"

print("NAME EXTRACTION TEST:")
for filename, text in cleaned_resumes.items():
    name = extract_name(text)
    print(f"{filename}: {name}")




NAME EXTRACTION TEST:
ml_engineer.txt: Anchit Shrivastava
data_analyst.txt: Priya Sharma
software_dev.txt: Rahul Kumar


In [8]:
def extract_email(resume_text):
  email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
  emails = re.findall(email_pattern,resume_text)
  return emails[0] if emails else "No email"
print("n\EMAIL EXTRACTION:")
for filename,text in cleaned_resumes.items():
  email = extract_email(text)
  print(f"{filename}: {email}")

n\EMAIL EXTRACTION:
ml_engineer.txt: anchit@email.com
data_analyst.txt: No email
software_dev.txt: No email


In [9]:
def extract_phone(resume_text):
  phone_pattern = [
      r'(\d{10})',
      r'(\+\d{2}\s?\d{10})',
      r'(\d{3} - \d{3}-\d{4})'
    ]

  for pattern in phone_pattern:
    phones = re.findall(pattern,resume_text)
    if phones:
      return phones[0]
  return "No phone"

print("\nPHONE EXTRACTION:")
for filename, text in cleaned_resumes.items():
  phone = extract_phone(text)
  print(f"{filename}: {phone}")



PHONE EXTRACTION:
ml_engineer.txt: No phone
data_analyst.txt: No phone
software_dev.txt: No phone


In [10]:
def parse_resume_complete(resume_text):
    result = {
        'name': extract_name(resume_text),
        'email': extract_email(resume_text),
        'phone': extract_phone(resume_text),
        'skills': extract_skills(resume_text, skills_list),
        'experience_years': extract_experience(resume_text),
        'skills_count': len(skills)
    }
    return result


print("\n" + "="*70)
print(" RESUME PARSER - DAY 2")
print("="*70)

for filename, text in cleaned_resumes.items():
    result = parse_resume_complete(text)
    print(f"\n{filename}:")
    for key, value in result.items():
        print(f"  {key.title()}: {value}")
    print("-" * 40)



 RESUME PARSER - DAY 2

ml_engineer.txt:
  Name: Anchit Shrivastava
  Email: anchit@email.com
  Phone: No phone
  Skills: ['python', 'sql', 'machine learning', 'nlp', 'scikit-learn', 'xgboost', 'pandas', 'power bi']
  Experience_Years: 3
  Skills_Count: 4
----------------------------------------

data_analyst.txt:
  Name: Priya Sharma
  Email: No email
  Phone: No phone
  Skills: ['python', 'sql', 'pandas', 'tableau', 'power bi']
  Experience_Years: 2
  Skills_Count: 4
----------------------------------------

software_dev.txt:
  Name: Rahul Kumar
  Email: No email
  Phone: No phone
  Skills: ['java', 'aws', 'docker', 'kubernetes']
  Experience_Years: 4
  Skills_Count: 4
----------------------------------------


In [11]:

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\W', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [12]:
job_description = """
We are looking for a Machine Learning Engineer.

Requirements:
- Strong Python skills
- Experience with Machine Learning and NLP
- Knowledge of SQL and Pandas
- Experience with deployment (Docker, AWS) """

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_similarity(resume_text,job_desc):
  tfidf = TfidfVectorizer()
  vectors = tfidf.fit_transform([resume_text,job_desc])

  similarity = cosine_similarity(vectors[0:1],vectors[1:2])

  return round(similarity[0][0] * 100,2)

In [14]:
print("\n" + "="*60)
print("MATCHING TEST (TF-IDF)")
print("="*60)

for filename, text in cleaned_resumes.items():
    score = calculate_similarity(text, job_description)
    print(f"{filename}: {score}% match")


MATCHING TEST (TF-IDF)
ml_engineer.txt: 11.66% match
data_analyst.txt: 9.34% match
software_dev.txt: 8.31% match


## ***Missing Skill Detection***

In [15]:
def get_missing_skills(resume_text, job_desc):
    resume_skills = extract_skills(resume_text, skills_list)
    job_skills = extract_skills(job_desc, skills_list)

    missing = [skill for skill in job_skills if skill not in resume_skills]

    return missing


In [16]:

for filename, text in cleaned_resumes.items():
    missing = get_missing_skills(text, job_description)

    print(f"\n{filename}:")
    print(f"Missing Skills: {missing}")


ml_engineer.txt:
Missing Skills: ['aws', 'docker']

data_analyst.txt:
Missing Skills: ['machine learning', 'nlp', 'aws', 'docker']

software_dev.txt:
Missing Skills: ['python', 'sql', 'machine learning', 'nlp', 'pandas']


## ***Final Analyzer Function***

In [17]:
def analyze_resume(resume_text, job_desc):

    parsed = parse_resume_complete(resume_text)

    match_score = calculate_similarity(resume_text, job_desc)
    missing_skills = get_missing_skills(resume_text, job_desc)

    result = {
        'name': parsed['name'],
        'email': parsed['email'],
        'experience': parsed['experience_years'],
        'skills': parsed['skills'],
        'match_score': match_score,
        'missing_skills': missing_skills
    }

    return result

In [18]:
print("\n" + "="*70)
print(" FINAL RESUME ANALYSIS")
print("="*70)

for filename, text in cleaned_resumes.items():
    result = analyze_resume(text, job_description)

    print(f"\n {filename}:")
    for key, value in result.items():
        print(f"{key}: {value}")



 FINAL RESUME ANALYSIS

 ml_engineer.txt:
name: Anchit Shrivastava
email: anchit@email.com
experience: 3
skills: ['python', 'sql', 'machine learning', 'nlp', 'scikit-learn', 'xgboost', 'pandas', 'power bi']
match_score: 11.66
missing_skills: ['aws', 'docker']

 data_analyst.txt:
name: Priya Sharma
email: No email
experience: 2
skills: ['python', 'sql', 'pandas', 'tableau', 'power bi']
match_score: 9.34
missing_skills: ['machine learning', 'nlp', 'aws', 'docker']

 software_dev.txt:
name: Rahul Kumar
email: No email
experience: 4
skills: ['java', 'aws', 'docker', 'kubernetes']
match_score: 8.31
missing_skills: ['python', 'sql', 'machine learning', 'nlp', 'pandas']


In [19]:
resume_df = pd.read_csv("/content/Resume_Data (2).csv")
job_df = pd.read_csv("/content/job_title_des.csv")

print(resume_df.head())
print(job_df.head())


job_df.columns = job_df.columns.str.strip()
resume_df.columns = resume_df.columns.str.strip()


   Resume_ID              Name                                        Skills  \
0          1        Ashley Ali                      TensorFlow, NLP, Pytorch   
1          2      Wesley Roman  Deep Learning, Machine Learning, Python, SQL   
2          3     Corey Sanchez         Ethical Hacking, Cybersecurity, Linux   
3          4  Elizabeth Carney                   Python, Pytorch, TensorFlow   
4          5        Julie Hill                              SQL, React, Java   

   Experience (Years) Education                Certifications  \
0                  10      B.Sc                           NaN   
1                  10       MBA                     Google ML   
2                   1       MBA  Deep Learning Specialization   
3                   7    B.Tech                 AWS Certified   
4                   4       PhD                           NaN   

                Job Role Recruiter Decision  Salary Expectation ($)  \
0          AI Researcher               Hire              

In [20]:
print(job_df.columns.tolist())
job_df.columns = job_df.columns.str.strip().str.replace('\xa0', '')
print(job_df.columns.tolist())
job_df['Job Description'].head()

['Unnamed: 0', 'Job Title', 'Job Description']
['Unnamed: 0', 'Job Title', 'Job Description']


,Job Description
0,We are looking for hire experts flutter develo...
1,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,JOB DESCRIPTION:\n\nStrong framework outside o...
4,job responsibility full stack engineer – react...


In [21]:
print(resume_df.columns)
print(job_df.columns)

job_df.columns = job_df.columns.str.strip()
resume_df.columns = resume_df.columns.str.strip()

resume_text = resume_df['Skills'] + " " + resume_df['Job Role']
job_text = job_df['Job Description']


resume_df['combined_text'] = (
    resume_df['Skills'] + " " +
    resume_df['Job Role'] + " " +
    resume_df['Education'] + " " +
    resume_df['Certifications']
)

resume_df['cleaned'] = resume_df['combined_text'].apply(clean_text)
job_df['cleaned'] = job_df['Job Description'].apply(clean_text)


for i in range(5):
  resume = resume_df['cleaned'][i]
  job = job_df['cleaned'][i]

  score = calculate_similarity(resume,job)

  print(f"\nResume {i}:{score}% match")


Index(['Resume_ID', 'Name', 'Skills', 'Experience (Years)', 'Education',
       'Certifications', 'Job Role', 'Recruiter Decision',
       'Salary Expectation ($)', 'Projects Count', 'AI Score (0-100)'],
      dtype='object')
Index(['Unnamed: 0', 'Job Title', 'Job Description'], dtype='object')

Resume 0:0.0% match

Resume 1:4.98% match

Resume 2:5.98% match

Resume 3:0.0% match

Resume 4:0.0% match


In [22]:

job_ml = job_df[job_df['Job Title'].str.contains("data|ml|python", case=False, na=False)]
if not job_ml.empty:
    job_text = job_ml.iloc[0]['cleaned']

    for i in range(5):
        resume = resume_df['cleaned'].iloc[i]
        score = calculate_similarity(resume, job_text)
        print(f"\nResume {i}: {score}% match")
else:
    print("No matching job found")


Resume 0: 0.0% match

Resume 1: 2.67% match

Resume 2: 0.49% match

Resume 3: 0.0% match

Resume 4: 0.0% match


## ***Skill Based Matching***

In [23]:
def hybrid_score(resume_text,job_text):
  tfidf_score = calculate_similarity(resume_text,job_text)

  resume_skills = extract_skills(resume_text,skills_list)
  job_skills = extract_skills(job_text, skills_list)

  skill_match = len(set(resume_skills) & set(job_skills))
  skill_score = (skill_match/ (len(job_skills) + 1)) * 100

  final_score = (0.7*tfidf_score) + (0.3 * skill_score)

  return round(final_score,2)

In [24]:
for i in range(5):
  resume = resume_df['cleaned'].iloc[i]

  score = hybrid_score(resume,job_text)
  print(f"\nResume {i}: {score}%match")


Resume 0: 0.0%match

Resume 1: 16.87%match

Resume 2: 0.34%match

Resume 3: 0.0%match

Resume 4: 0.0%match


In [25]:
!pip install sentence-transformers

In [26]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode ALL resumes at once
resume_embeddings = model.encode(
    resume_df['cleaned'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [27]:
from sentence_transformers import util

job_embedding = model.encode(job_text, convert_to_tensor=True)

scores = util.cos_sim(resume_embeddings, job_embedding)

scores = scores.cpu().numpy().flatten()

resume_df['bert_score'] = scores * 100



In [28]:
def fast_final_ranking(resume_df, job_text):

    # Encode resumes
    resume_embeddings = model.encode(
        resume_df['cleaned'].tolist(),
        convert_to_tensor=True
    )

    # Encode job
    job_embedding = model.encode(job_text, convert_to_tensor=True)

    # BERT scores
    scores = util.cos_sim(resume_embeddings, job_embedding)
    scores = scores.cpu().numpy().flatten() * 100

    resume_df['bert_score'] = scores

    return resume_df.sort_values(by='bert_score', ascending=False)

In [29]:
ranked = resume_df.sort_values(by='bert_score', ascending=False)

ranked[['Name', 'Job Role', 'bert_score']].head(5)

,Name,Job Role,bert_score
4385,Debra Tapia,Software Engineer,45.311852
3028,Elizabeth Garcia,Software Engineer,44.664433
4105,Renee Vasquez,Software Engineer,43.940308
1034,Anthony Perez,Software Engineer,43.703251
2936,James Smith,Software Engineer,43.623672


### ***Enriched Result***


In [30]:
resume_df = resume_df.fillna('')
job_df = job_df.fillna('')

resume_df.columns = resume_df.columns.str.strip()
job_df.columns = job_df.columns.str.strip()

In [31]:
resume_df['combined_text'] = (
    resume_df['Skills'] + " " +
    resume_df['Job Role'] + " " +
    resume_df['Education'] + " " +
    resume_df['Certifications']
)

In [32]:
job_df['combined'] = (
    job_df['Job Title'] + " " +
    job_df['Job Description']
)

resume_df['cleaned'] = resume_df['combined_text'].apply(clean_text)
job_df['cleaned'] = job_df['combined'].apply(clean_text)

resume_df['cleaned'] = resume_df['cleaned'].astype(str)
job_df['cleaned'] = job_df['cleaned'].astype(str)

In [33]:
job_ml = job_df[job_df['Job Title'].str.contains("machine learning|data scientist", case=False, na=False)]

job_text = job_ml.iloc[0]['cleaned']


resume_embeddings = model.encode(
    resume_df['cleaned'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True
)

job_embedding = model.encode(job_text, convert_to_tensor=True)

scores = util.cos_sim(resume_embeddings, job_embedding)
scores = scores.cpu().numpy().flatten() * 100

resume_df['bert_score'] = scores

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [34]:
ranked = resume_df.sort_values(by='bert_score', ascending=False)

ranked[['Name', 'Job Role', 'bert_score']].head(5)

,Name,Job Role,bert_score
2573,James Alvarez,Software Engineer,60.548126
2191,Joel Hayes,Software Engineer,59.581505
2741,Christopher Maynard,Software Engineer,58.115185
2331,David Navarro,Software Engineer,57.778275
3365,Sydney Shelton,Software Engineer,57.557552


## ***Streamlit Deployment***

In [35]:
!pip install streamlit